<a href="https://colab.research.google.com/github/jackpang-ltp/MathModeling/blob/main/IMMC_optimization_lab_evcharging_project_forstudents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 33.7 MB/s eta 0:00:00


Set up

In [18]:
import math
from pathlib import Path

import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt
import pandas as pd

NUM_CHARGERS = 4
H = 20
DELTA = 30
START_HOUR = 8
CHARGER_POWER_KW = 22
MAX_WAIT_MINUTES = 120

EMBEDDED_DATA = [
  {
    "ev_id": "EV01",
    "arrival_time": "08:00",
    "arrival_slot": 0,
    "energy_demand_kwh": 38,
    "charging_slots": 4,
    "charging_duration_min": 120,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV02",
    "arrival_time": "08:00",
    "arrival_slot": 0,
    "energy_demand_kwh": 41,
    "charging_slots": 4,
    "charging_duration_min": 120,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV03",
    "arrival_time": "08:00",
    "arrival_slot": 0,
    "energy_demand_kwh": 29,
    "charging_slots": 3,
    "charging_duration_min": 90,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV04",
    "arrival_time": "08:00",
    "arrival_slot": 0,
    "energy_demand_kwh": 31,
    "charging_slots": 3,
    "charging_duration_min": 90,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV05",
    "arrival_time": "08:30",
    "arrival_slot": 1,
    "energy_demand_kwh": 36,
    "charging_slots": 4,
    "charging_duration_min": 120,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV06",
    "arrival_time": "08:30",
    "arrival_slot": 1,
    "energy_demand_kwh": 42,
    "charging_slots": 4,
    "charging_duration_min": 120,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV07",
    "arrival_time": "08:30",
    "arrival_slot": 1,
    "energy_demand_kwh": 28,
    "charging_slots": 3,
    "charging_duration_min": 90,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV08",
    "arrival_time": "09:00",
    "arrival_slot": 2,
    "energy_demand_kwh": 30,
    "charging_slots": 3,
    "charging_duration_min": 90,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV09",
    "arrival_time": "09:00",
    "arrival_slot": 2,
    "energy_demand_kwh": 9,
    "charging_slots": 1,
    "charging_duration_min": 30,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV10",
    "arrival_time": "09:00",
    "arrival_slot": 2,
    "energy_demand_kwh": 10,
    "charging_slots": 1,
    "charging_duration_min": 30,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV11",
    "arrival_time": "09:30",
    "arrival_slot": 3,
    "energy_demand_kwh": 8,
    "charging_slots": 1,
    "charging_duration_min": 30,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV12",
    "arrival_time": "09:30",
    "arrival_slot": 3,
    "energy_demand_kwh": 7,
    "charging_slots": 1,
    "charging_duration_min": 30,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV13",
    "arrival_time": "10:00",
    "arrival_slot": 4,
    "energy_demand_kwh": 18,
    "charging_slots": 2,
    "charging_duration_min": 60,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV14",
    "arrival_time": "10:00",
    "arrival_slot": 4,
    "energy_demand_kwh": 21,
    "charging_slots": 2,
    "charging_duration_min": 60,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV15",
    "arrival_time": "10:30",
    "arrival_slot": 5,
    "energy_demand_kwh": 10,
    "charging_slots": 1,
    "charging_duration_min": 30,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV16",
    "arrival_time": "10:30",
    "arrival_slot": 5,
    "energy_demand_kwh": 9,
    "charging_slots": 1,
    "charging_duration_min": 30,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV17",
    "arrival_time": "11:00",
    "arrival_slot": 6,
    "energy_demand_kwh": 30,
    "charging_slots": 3,
    "charging_duration_min": 90,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV18",
    "arrival_time": "11:00",
    "arrival_slot": 6,
    "energy_demand_kwh": 20,
    "charging_slots": 2,
    "charging_duration_min": 60,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV19",
    "arrival_time": "15:00",
    "arrival_slot": 14,
    "energy_demand_kwh": 17,
    "charging_slots": 2,
    "charging_duration_min": 60,
    "departure_time": "18:00",
    "departure_slot": 20
  },
  {
    "ev_id": "EV20",
    "arrival_time": "16:00",
    "arrival_slot": 16,
    "energy_demand_kwh": 27,
    "charging_slots": 3,
    "charging_duration_min": 90,
    "departure_time": "18:00",
    "departure_slot": 20
  }
]

csv_path = Path("ev_charging_20_ev.csv")
if csv_path.exists():
    ev = pd.read_csv(csv_path)
else:
    ev = pd.DataFrame(EMBEDDED_DATA).head(5) # Further reduced the number of EVs

def slot_to_clock(slot):
    minutes = START_HOUR * 60 + DELTA * int(slot)
    return f"{minutes // 60:02d}:{minutes % 60:02d}"

ev

,ev_id,arrival_time,arrival_slot,energy_demand_kwh,charging_slots,charging_duration_min,departure_time,departure_slot
0,EV01,08:00,0,38,4,120,18:00,20
1,EV02,08:00,0,41,4,120,18:00,20
2,EV03,08:00,0,29,3,90,18:00,20
3,EV04,08:00,0,31,3,90,18:00,20
4,EV05,08:30,1,36,4,120,18:00,20
5,EV06,08:30,1,42,4,120,18:00,20
6,EV07,08:30,1,28,3,90,18:00,20
7,EV08,09:00,2,30,3,90,18:00,20
8,EV09,09:00,2,9,1,30,18:00,20
9,EV10,09:00,2,10,1,30,18:00,20


In [3]:
energy_per_slot = CHARGER_POWER_KW * DELTA / 60
calculated_slots = (ev["energy_demand_kwh"] / energy_per_slot).apply(math.ceil)

assert len(ev) == 20
assert ev["ev_id"].is_unique
assert (calculated_slots == ev["charging_slots"]).all()
assert (ev["arrival_slot"] + ev["charging_slots"] <= ev["departure_slot"]).all()

total_charge_slots = int(ev["charging_slots"].sum())
hub_capacity_slots = NUM_CHARGERS * H
print(f"Energy per slot: {energy_per_slot:.1f} kWh")
print(f"Total charging work: {total_charge_slots} charger-slots")
print(f"Hub capacity: {hub_capacity_slots} charger-slots")
print(f"Whole-day utilization: {total_charge_slots / hub_capacity_slots:.1%}")


Energy per slot: 11.0 kWh
Total charging work: 48 charger-slots
Hub capacity: 80 charger-slots
Whole-day utilization: 60.0%


FCFS Baseline

In [4]:
def fcfs_schedule(data, num_chargers=NUM_CHARGERS):
    """Schedule EVs by (arrival_slot, ev_id) on the earliest available charger."""
    availability = [0] * num_chargers
    records = []

    ordered = data.sort_values(["arrival_slot", "ev_id"]).reset_index(drop=True)
    for row in ordered.itertuples(index=False):
        charger = min(
            range(num_chargers),
            key=lambda c: (max(availability[c], row.arrival_slot), c),
        )
        start = max(availability[charger], row.arrival_slot)
        end = start + row.charging_slots
        if end > row.departure_slot:
            raise ValueError(f"FCFS cannot complete {row.ev_id} by departure.")

        availability[charger] = end
        records.append(
            {
                "ev_id": row.ev_id,
                "charger": charger + 1,
                "arrival_slot": row.arrival_slot,
                "arrival_time": slot_to_clock(row.arrival_slot),
                "start_slot": start,
                "start_time": slot_to_clock(start),
                "end_slot": end,
                "end_time": slot_to_clock(end),
                "charging_slots": row.charging_slots,
                "wait_min": (start - row.arrival_slot) * DELTA,
            }
        )

    return pd.DataFrame(records).sort_values(["charger", "start_slot", "ev_id"])


fcfs = fcfs_schedule(ev)
fcfs.head()


,ev_id,charger,arrival_slot,arrival_time,start_slot,start_time,end_slot,end_time,charging_slots,wait_min
0,EV01,1,0,08:00,0,08:00,4,10:00,4,0
6,EV07,1,1,08:30,4,10:00,7,11:30,3,90
8,EV09,1,2,09:00,7,11:30,8,12:00,1,150
12,EV13,1,4,10:00,8,12:00,10,13:00,2,120
18,EV19,1,14,15:00,14,15:00,16,16:00,2,0


## MILP Formulation

We define a Mixed-Integer Linear Programming (MILP) model to optimize the EV charging schedule. The goal is to minimize the total waiting time for all electric vehicles, subject to constraints such as charger availability, each EV being charged, and EVs departing on time.

In [19]:
model = gp.Model("ev_charging_scheduler")

# Sets
EVs = ev["ev_id"].tolist()
Chargers = range(NUM_CHARGERS)
TimeSlots = range(H) # H is the total number of time slots available

# Parameters
ArrivalSlot = ev.set_index("ev_id")["arrival_slot"]
ChargingSlots = ev.set_index("ev_id")["charging_slots"]
DepartureSlot = ev.set_index("ev_id")["departure_slot"]

In [20]:
# Decision Variables

# x[i, c, t] = 1 if EV i is assigned to charger c and starts charging at time slot t
x = model.addVars(
    EVs, Chargers, TimeSlots, vtype=GRB.BINARY, name="charge_assignment"
)

# w[i] = waiting time in minutes for EV i
w = model.addVars(EVs, vtype=GRB.CONTINUOUS, name="wait_time")

# s[i, t] = 1 if EV i starts charging at time slot t
s = model.addVars(EVs, TimeSlots, vtype=GRB.BINARY, name="start_charging")

In [21]:
# Objective: Minimize total waiting time
model.setObjective(w.sum(), GRB.MINIMIZE)

In [22]:
# Constraints

# 1. Each EV must be charged exactly once
model.addConstrs(
    (gp.quicksum(x[i, c, t] for c in Chargers for t in TimeSlots if t >= ArrivalSlot[i] and t + ChargingSlots[i] <= DepartureSlot[i]) == 1
     for i in EVs), name="each_ev_charged"
)

# 2. Link x and s variables: s[i, t] is 1 if EV i starts charging at time t on any charger
model.addConstrs(
    (gp.quicksum(x[i, c, t] for c in Chargers) == s[i, t]
     for i in EVs for t in TimeSlots), name="link_x_s"
)

# 3. Charger capacity: A charger can only charge one EV at a time
for c in Chargers:
    for t_prime in TimeSlots:
        model.addConstr(
            gp.quicksum(
                x[i, c, t] for i in EVs for t in TimeSlots
                if t <= t_prime < t + ChargingSlots[i] # If EV i is charging at t_prime
            ) <= 1, name=f"charger_capacity_c{c}_t{t_prime}"
        )

# 4. Wait time calculation
model.addConstrs(
    (w[i] == gp.quicksum(t * s[i, t] for t in TimeSlots) - ArrivalSlot[i]
     for i in EVs), name="wait_time_calc"
)

# 5. Charging must respect arrival and departure windows (implicitly handled by variable domains and constraints 1)
# and the summation in constraint 1 for `x` variable domain.

print("MILP model defined successfully.")

MILP model defined successfully.


In [23]:
# Optimize the model
model.optimize()

print(f"Optimization status: {model.status}")

if model.status == GRB.OPTIMAL:
    print(f"Optimal objective value (total wait time): {model.objVal * DELTA} minutes")
elif model.status == GRB.INFEASIBLE:
    print("Model is infeasible.")
elif model.status == GRB.UNBOUNDED:
    print("Model is unbounded.")
else:
    print(f"Optimization stopped with status {model.status}")

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 300 rows, 1010 columns and 4140 nonzeros (Min)
Model fingerprint: 0xcb374133
Model has 10 linear objective coefficients
Variable types: 10 continuous, 1000 integer (1000 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]

Found heuristic solution: objective 42.0000000
Presolve removed 94 rows and 170 columns
Presolve time: 0.02s
Presolved: 206 rows, 840 columns, 3406 nonzeros
Found heuristic solution: objective 41.0000000
Variable types: 0 continuous, 840 integer (834 binary)

Root relaxation: objective 1.300000e+01, 196 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bound

In [24]:
# Extract and display solution
if model.status == GRB.OPTIMAL:
    milp_records = []
    for i in EVs:
        start_slot = None
        charger = None
        for t in TimeSlots:
            for c in Chargers:
                if x[i, c, t].X > 0.5: # If x[i,c,t] is 1
                    start_slot = t
                    charger = c
                    break
            if start_slot is not None: # Break outer loop once found
                break

        if start_slot is not None:
            end_slot = start_slot + ChargingSlots[i]
            milp_records.append(
                {
                    "ev_id": i,
                    "charger": charger + 1,
                    "arrival_slot": ArrivalSlot[i].item(), # .item() to get scalar
                    "arrival_time": slot_to_clock(ArrivalSlot[i].item()),
                    "start_slot": start_slot,
                    "start_time": slot_to_clock(start_slot),
                    "end_slot": end_slot,
                    "end_time": slot_to_clock(end_slot),
                    "charging_slots": ChargingSlots[i].item(),
                    "wait_min": w[i].X * DELTA,
                }
            )
        else:
            print(f"Warning: EV {i} not scheduled.")

    milp_schedule = pd.DataFrame(milp_records).sort_values(
        ["charger", "start_slot", "ev_id"]
    )
    display(milp_schedule.head())
else:
    print("No optimal solution found to display.")

,ev_id,charger,arrival_slot,arrival_time,start_slot,start_time,end_slot,end_time,charging_slots,wait_min
1,EV02,1,0,08:00,0,08:00,4,10:00,4,0.0
6,EV07,1,1,08:30,4,10:00,7,11:30,3,90.0
0,EV01,2,0,08:00,0,08:00,4,10:00,4,0.0
4,EV05,2,1,08:30,4,10:00,8,12:00,4,90.0
2,EV03,3,0,08:00,0,08:00,3,09:30,3,0.0


MILP formulation